In [ ]:
#  pip install deap scikit-learn graphviz pydotplus matplotlib seaborn
# !pip install -U scikit-learn imbalanced-learn


# Number of features
top_n = 15

# Import Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import export_graphviz
import pydotplus
import os
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image
from deap import base, creator, tools, algorithms
import random
from imblearn.over_sampling import SMOTE
from collections import Counter

# Random Forest Code

#### This line reads the Excel file named 'Random Forest Input.xlsx' into a pandas DataFrame called data.  

In [ ]:
import pandas as pd

# List of file names
file_name = 'NonEnsemble-Enhanced Random Forest Input.xlsx'
# file_name = 'Random Forest Input.xlsx'
# file_name = 'Enhanced Random Forest Input.xlsx'

# Load the DataFrame using the selected file name
data = pd.read_excel(file_name)

# Now 'data' contains the DataFrame from the selected file

# Edit the "Patient ID" column to remove everything after the first "_"
data['Patient ID'] = data['Patient ID'].str.split('_').str[0]

# Optional:

In [ ]:
numeric_data=data


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import random

# Define target and features
target = numeric_data['Hemorrhage Level'].astype(str)

# Drop columns with the word "title" in their name
features = numeric_data.drop(columns=['Hemorrhage Level', 'Patient ID'])

# Drop columns that contain the word "title"
features = features.loc[:, ~features.columns.str.contains('title', case=False)]

# Drop columns where data cannot be converted to float
features = features.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='any')

# Custom selection logic for test folders
hemorrhage_10_percent = [
    "ERNE10", "ERNE12", "ERNE23", "ERNE25", "ERNE27", "ERNE32", "ERNE33", "ERNE38", "ERNE39", "ERNE40", 
    "ERNE44", "ERNE48", "ERNE58", "ERNE59", "ERNE61", "ERNE62", "ERNE67", "ERNE70", "ERNE91", "ERNE92", 
    "ERNE93", "ERNE94", "ERNE95", "ERNE96", "ERNE97", "ERNE98", "ERNE99", "ERNE100", "ERNE101", "ERNE102", 
    "ERNE103", "ERNE104", "ERNE105", "ERNE106", "ERNE107", "ERNE108", "ERNE109", "ERNE110", "ERNE111", 
    "ERNE112", "ERNE113", "ERNE114", "ERNE115", "ERNE116", "ERNE117", "ERNE118", "ERNE119", "ERNE120", 
    "ERNE121", "ERNE122", "ERNE123", "ERNE124", "ERNE125", "ERNE126"
]

hemorrhage_20_percent = [
    "ERNE19", "ERNE21", "ERNE30", "ERNE34", "ERNE41", "ERNE45", "ERNE47", "ERNE49", "ERNE50", "ERNE51", 
    "ERNE53", "ERNE54", "ERNE56", "ERNE60", "ERNE63", "ERNE69", "ERNE71", "ERNE72", "ERNE73", "ERNE74", 
    "ERNE75", "ERNE76", "ERNE77", "ERNE78", "ERNE79", "ERNE80", "ERNE81", "ERNE82", "ERNE83", "ERNE84", 
    "ERNE85", "ERNE86", "ERNE87", "ERNE88", "ERNE89", "ERNE90"
]

hemorrhage_30_percent = [
    "ERNE2", "ERNE5", "ERNE7", "ERNE14", "ERNE20", "ERNE26", "ERNE29", "ERNE31", "ERNE35", "ERNE36", 
    "ERNE43", "ERNE52", "ERNE55", "ERNE57", "ERNE64", "ERNE65", "ERNE66", "ERNE68"
]

In [ ]:
size_10=len(hemorrhage_10_percent)
size_20=len(hemorrhage_20_percent)
size_30=len(hemorrhage_30_percent)

In [ ]:
train_sizes=[1/3*size_10,1/3*size_20,1/3*size_30]

In [ ]:
np.min(np.array(train_sizes))

In [ ]:

# Custom parameters for selection
test_amount = 6


# Function to randomly select strings from a given list
def select_random_strings(data_list, number_of_strings):
    return random.sample(data_list, number_of_strings)

# Initialize the lists to store the results
test_folders = []
validation_folders = []

# Perform the selection for test_folders
selected_10 = select_random_strings(hemorrhage_10_percent, test_amount)
test_folders.extend(selected_10)
hemorrhage_10_percent = [item for item in hemorrhage_10_percent if item not in selected_10]

selected_20 = select_random_strings(hemorrhage_20_percent, test_amount)
test_folders.extend(selected_20)
hemorrhage_20_percent = [item for item in hemorrhage_20_percent if item not in selected_20]

selected_30 = select_random_strings(hemorrhage_30_percent, test_amount)
test_folders.extend(selected_30)
hemorrhage_30_percent = [item for item in hemorrhage_30_percent if item not in selected_30]



# Select indices for training and testing
train_indices = numeric_data[~numeric_data['Patient ID'].isin(test_folders)].index
test_indices = numeric_data[numeric_data['Patient ID'].isin(test_folders)].index

# Split the data based on the custom indices
X_train, X_test = features.loc[train_indices], features.loc[test_indices]
y_train, y_test = target.loc[train_indices], target.loc[test_indices]

# Proceed with model training using X_train, y_train and testing with X_test, y_test


In [ ]:
# Display selected test folders
print("Test Folders:", test_folders)


## Preprocessing
#### This line filters the DataFrame to retain only columns with numeric data types, which are needed for most machine learning algorithms.

In [ ]:
# Drop non-numeric columns and rows
numeric_data = data.select_dtypes(include=[np.number])

## Convert categorical columns to numeric columns


categorical_cols: Selects columns with object (typically string) data types.


label_encoders: Initializes an empty dictionary to store the label encoders for each column.


The for loop iterates over each categorical column, applies LabelEncoder to convert categories to numeric codes, and updates the numeric_data DataFrame.

In [ ]:
# Encode categorical columns
categorical_cols = data.select_dtypes(include=[object])
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    numeric_data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

## Define the "target" i.e data label and features that aid in classification


target: Assigns the 'Hemorrhage Level' column as the target variable. In the context of random Forest, this is essentially the data label for my dataset.


features: After dropping the 'Hemorrhage Level' and 'Title' (unique animal identifier) columns from the dataset, only the traditional waveform parameters are available.

In [ ]:
# hemorrhage_10_percent = [
#     "ERNE10", "ERNE12", "ERNE23", "ERNE25", "ERNE27", "ERNE32", "ERNE33", "ERNE38", "ERNE39", "ERNE40", 
#     "ERNE44", "ERNE48", "ERNE58", "ERNE59", "ERNE61", "ERNE62", "ERNE67", "ERNE70", "ERNE91", "ERNE92", 
#     "ERNE93", "ERNE94", "ERNE95", "ERNE96", "ERNE97", "ERNE98", "ERNE99", "ERNE100", "ERNE101", "ERNE102", 
#     "ERNE103", "ERNE104", "ERNE105", "ERNE106", "ERNE107", "ERNE108", "ERNE109", "ERNE110", "ERNE111", 
#     "ERNE112", "ERNE113", "ERNE114", "ERNE115", "ERNE116", "ERNE117", "ERNE118", "ERNE119", "ERNE120", 
#     "ERNE121", "ERNE122", "ERNE123", "ERNE124", "ERNE125", "ERNE126"
# ]

# hemorrhage_20_percent = [
#     "ERNE19", "ERNE21", "ERNE30", "ERNE34", "ERNE41", "ERNE45", "ERNE47", "ERNE49", "ERNE50", "ERNE51", 
#     "ERNE53", "ERNE54", "ERNE56", "ERNE60", "ERNE63", "ERNE69", "ERNE71", "ERNE72", "ERNE73", "ERNE74", 
#     "ERNE75", "ERNE76", "ERNE77", "ERNE78", "ERNE79", "ERNE80", "ERNE81", "ERNE82", "ERNE83", "ERNE84", 
#     "ERNE85", "ERNE86", "ERNE87", "ERNE88", "ERNE89", "ERNE90"
# ]

# hemorrhage_30_percent = [
#     "ERNE2", "ERNE5", "ERNE7", "ERNE14", "ERNE20", "ERNE26", "ERNE29", "ERNE31", "ERNE35", "ERNE36", 
#     "ERNE43", "ERNE52", "ERNE55", "ERNE57", "ERNE64", "ERNE65", "ERNE66", "ERNE68"
# ]


# test_amount=1
# val_amount=1
# # Function to randomly select four strings from a given list
# def select_random_strings(data_list, number_of_strings):
#     return random.sample(data_list, number_of_strings)

# # Initialize the lists to store the results
# test_folders = []
# validation_folders = []

# # Perform the selection for test_folders
# selected_10 = select_random_strings(hemorrhage_10_percent,test_amount)
# test_folders.extend(selected_10)
# hemorrhage_10_percent = [item for item in hemorrhage_10_percent if item not in selected_10]

# selected_20 = select_random_strings(hemorrhage_20_percent,test_amount)
# test_folders.extend(selected_20)
# hemorrhage_20_percent = [item for item in hemorrhage_20_percent if item not in selected_20]

# selected_30 = select_random_strings(hemorrhage_30_percent,test_amount)
# test_folders.extend(selected_30)
# hemorrhage_30_percent = [item for item in hemorrhage_30_percent if item not in selected_30]

# # Print the results
# print("Test Folders:", test_folders)

# # Concatenate the DataFrames along the channel axis
# X_list = [df.iloc[:, :-3].values[:, :, np.newaxis] for df in dfs]  # Convert each DataFrame to numpy array and add a new axis
# # y = np.dstack([df["Hemorrhage Level"].values for df in dfs])  # Stack the target values along the channel axis
# # y = np.concatenate([df["Hemorrhage Level"].values[:, np.newaxis] for df in dfs], axis=1)
# y = np.concatenate([df["Hemorrhage Level"].values[:, np.newaxis] for df in dfs], axis=1)
# # Select the "Hemorrhage Level" values from the first DataFrame
# y = dfs[0]["Hemorrhage Level"].values[:, np.newaxis]

# # Now, y will have a single column representing the "Hemorrhage Level" values across all sheets
# # Assuming y_train is your label data
# classes = len(np.unique(y))
# print("Number of unique classes:", classes)

# unique_values = np.unique(y)
# print("Unique values:", unique_values)


# unique_values, counts = np.unique(y, return_counts=True)
# for value, count in zip(unique_values, counts):
#     print("Value:", value, "- Count:", count)
    
# # Split data indices based on folders
# train_indices = np.concatenate([df[~(df['Parent Folder'].isin(test_folders) | df['Parent Folder'].isin(validation_folders))].index.values for df in dfs])
# test_indices = np.concatenate([df[df['Parent Folder'].isin(test_folders)].index.values for df in dfs])

# print("Train Indices:", train_indices)

# print("Test Indices:", test_indices)


## Split the Data into train and test data
Using the function "train_test_split" the features and targets are divided into training and testing sets. Here I partitioned 20% of the data for testing. The random_state parameter ensures reproducibility.

In [ ]:
# # Define target and features
# target = numeric_data['Hemorrhage Level'] 

# # Drop columns with the word "title" in their name
# features = numeric_data.drop(columns=['Hemorrhage Level', 'Patient ID'])

# # Drop columns that contain the word "title"
# features = features.loc[:, ~features.columns.str.contains('title', case=False)]

# # Drop columns where data cannot be converted to float
# features = features.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='any')


# # Split the data
# X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.4, stratify=target,random_state=42)

In [ ]:
numeric_data.head()

## Balance Classes

In [ ]:
# Class balancing with SMOTE on training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Calculate the size of the smallest class in the test set
counter = Counter(y_test)
min_class_size = min(counter.values())

# Drop elements from overrepresented classes in the test set
X_test_balanced = pd.DataFrame()
y_test_balanced = []
for cls, count in counter.items():
    indices = np.where(y_test == cls)[0]
    if count > min_class_size:
        drop_indices = np.random.choice(indices, size=count - min_class_size, replace=False)
        X_test_balanced = pd.concat([X_test_balanced, X_test.drop(X_test.index[drop_indices]).reset_index(drop=True)])
        y_test_balanced.extend(y_test.drop(y_test.index[drop_indices]))
    else:
        X_test_balanced = pd.concat([X_test_balanced, X_test.iloc[indices]])
        y_test_balanced.extend(y_test.iloc[indices])

# Convert y_test_balanced to numpy array
y_test_balanced = np.array(y_test_balanced)

# Trim to the size of the smallest class
counter_balanced = Counter(y_test_balanced)
min_class_size_balanced = min(counter_balanced.values())

# Ensure close to the minimum class size samples per class in the test set
X_test_balanced_trimmed = pd.DataFrame()
y_test_balanced_trimmed = []
desired_class_size = min(min(features.shape[0]/4, min_class_size_balanced), min_class_size)  # Example: Close to 4 samples per class or the minimum class size

for cls, count in counter_balanced.items():
    indices = np.where(y_test_balanced == cls)[0]
    np.random.shuffle(indices)  # shuffle indices
    keep_indices = indices[:min(desired_class_size, count)]  # keep up to desired_class_size samples or less if count < desired_class_size
    X_test_balanced_trimmed = pd.concat([X_test_balanced_trimmed, X_test_balanced.iloc[keep_indices]])
    y_test_balanced_trimmed.extend(y_test_balanced[keep_indices])


## Train a Random Forest model.

initial_rf: Creates a RandomForestClassifier starting with 100 trees.


fit: Trains the model on the training data (X_train, y_train) that was previously defined.

### Random Forest Hyperparameters

n_estimators: The number of trees in the forest.

max_depth: The maximum depth of each tree.

min_samples_split: The minimum number of samples required to split an internal node.

min_samples_leaf: The minimum number of samples required to be at a leaf node.

bootstrap: Whether bootstrap samples are used when building trees.

random_state: Seed used by the random number generator.

#Define hyperparameters
n_estimators = 100
max_depth = 10
min_samples_split = 2
min_samples_leaf = 1

bootstrap = True
random_state = 42

In [ ]:
# Define hyperparameters
n_estimators = 100
max_depth = top_n
min_samples_split = 2
min_samples_leaf = 1
bootstrap = True
random_state = 42

# Initialize the RandomForestClassifier with the defined hyperparameters
initial_rf = RandomForestClassifier(n_estimators=n_estimators,
                                     max_depth=max_depth,
                                     min_samples_split=min_samples_split,
                                     min_samples_leaf=min_samples_leaf,
                                     bootstrap=bootstrap,
                                     random_state=random_state)

# Fit the model to the training data
initial_rf.fit(X_train, y_train)

## Genetic Algorithms 101

Theoretical Perspective
Initial Population:
The evolutionary algorithm starts with an initial population of individuals. Each individual is a candidate solution from the original Random Forest.

Fitness Evaluation:
Each “individual” is evaluated using the fitness function to assign a fitness value which is how that solution works with classification.

Selection:
Individuals are selected based on their fitness values. Higher fitness increases the chance of being selected.


Crossover and Mutation:
Selected individuals undergo crossover (recombination) and mutation to produce offspring, which form the next generation that is then fed back into the evaluation step.


Iteration:
This process repeats for many generations, with the goal of evolving increasingly better solutions or, in other words, “better fitness.”


Termination:
The algorithm terminates after a fixed number of generations or when a satisfactory fitness level is achieved.

The final output is essentially a pool of “chromosomes” that are ideal for solving a problem.


## Evaluate subsets of the Random Forest trees 

The "evaluate" function assesses the quality of a given 'individual' (i.e. a potential solution) in the context of the evolutionary algorithm. Each individual, which is inputted into the function, represents a specific selection of trees from the initial unedited Random Forest code. This function works to evaluate how well the selected subset of trees performs by measuring the accuracy of the pruned Random Forest on the test data.



individual: Represents a binary vector indicating which trees to keep.


selected_trees: Subsets the trees in initial_rf based on individual.


If no trees are selected, it returns an accuracy of 0.


Creates a new RandomForestClassifier with the selected trees and evaluates its accuracy on the test set.

In this context, i.e. of evolutionary algorithms and genetic programming, the accuracy is represented by a "fitness value" which is in its essence the a numerical representation of how well a particular solution true (i.e. "individual") performs in solving the given problem. 

In [ ]:
# Extract feature importances and select the top 10 features
importances = initial_rf.feature_importances_
indices = np.argsort(importances)[::-1]  # Sort in descending order
top_10_indices = indices[:top_n]  # Select top 10 features

# Subset the data to include only the top 10 important features
# Use iloc to index rows and columns by their integer location
X_train_top10 = X_train.iloc[:, top_10_indices]
X_test_top10 = X_test.iloc[:, top_10_indices]

# Redefine the initial random forest using only the top 10 important features
initial_rf.fit(X_train_top10, y_train)

# Define the evaluation function
def evaluate(individual):
    selected_trees = [tree for i, tree in enumerate(initial_rf.estimators_) if individual[i] == 1]

    if not selected_trees:
        return 0,

    # Create a new Random Forest with the selected trees
    rf = RandomForestClassifier(n_estimators=len(selected_trees), random_state=42)
    rf.estimators_ = selected_trees
    rf.classes_ = initial_rf.classes_
    rf.n_classes_ = initial_rf.n_classes_
    rf.n_outputs_ = initial_rf.n_outputs_

    # Evaluate the new forest
    y_pred = rf.predict(X_test_top10)  # Use the test data with top 10 features
    return accuracy_score(y_test, y_pred),


## Define the genetic algorithm components using DEAP

DEAP (Distributed Evolutionary Algorithms in Python) is a library used for implementing evolutionary algorithms. It provides tools to create and manipulate genetic algorithms, like the creation of custom data types for individuals and their fitness.

### Line 1: creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create:

"creator" is a module in DEAP that allows you to create new classes at runtime. "create" is a method of the creator module used to define new types of objects.

"FitnessMax":

This is the name of the new class being created. 


base.Fitness:


base.Fitness is the base class that the new FitnessMax class will inherit from.


weights=(1.0,):
The weights parameter specifies the type of optimization: maximization or minimization.
A positive value (1.0) means that the fitness will be maximized.
If we had used a negative value (-1.0), it would mean the fitness is to be minimized.
The weights attribute can be a tuple with multiple values for multi-objective optimization. Here, a single objective is being maximized.




### Line 2: creator.create("Individual", list, fitness=creator.FitnessMax)

"Individual":

This is the name of the new class being created.


list:
The new Individual class will inherit from Python's built-in list class.This means that an Individual will behave like a list.

fitness=creator.FitnessMax:

This specifies that each Individual will have a fitness attribute. This refers to an instance of the FitnessMax class created earlier, which connects the individual to its fitness evaluation, allowing the algorithm to assess the quality of the solution it represents.

In [ ]:
# DEAP setup remains the same
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

## Register functions for creating individuals and populations.


attr_bool: Generates a random binary value (0 or 1).


individual: Creates an individual composed of binary attributes (representing trees).


population: Creates a population of individuals.

In [ ]:
toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, n=len(initial_rf.estimators_))
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

## Register genetic algorithm operators.
mate: Two-point crossover.

mutate: Bit-flip mutation with probability 0.05.

select: Tournament selection with size 3.

evaluate: Evaluation function defined earlier.

In [ ]:
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", evaluate)

## Define parameters for the genetic algorithm.


population: Initializes the population with 50 individuals.


NGEN: Number of generations to evolve.


CXPB: Crossover probability.


MUTPB: Mutation probability.

In [ ]:
# Genetic Algorithm parameters
population = toolbox.population(n=50)
NGEN = 30
CXPB, MUTPB = 0.5, 0.2

## Run the genetic algorithm to evolve the population.

Iterates through the number of generations (NGEN).


Generates offspring using crossover and mutation.


Evaluates the offspring and updates their fitness.


Selects the next generation of individuals.


Prints the best accuracy of the current generation.

In [ ]:
# # Run Genetic Algorithm
# for gen in range(NGEN):
#     offspring = algorithms.varAnd(population, toolbox, cxpb=CXPB, mutpb=MUTPB)
#     fits = map(toolbox.evaluate, offspring)
#     for fit, ind in zip(fits, offspring):
#         ind.fitness.values = fit
#     population = toolbox.select(offspring, k=len(population))
#     print(f"Generation {gen} - Best Accuracy: {max(ind.fitness.values for ind in population)}")


# Genetic Algorithm Hyperparameters

In [ ]:
# GA Parameters
NGEN = 50  # Number of generations
pop_size = 100  # Population size
CXPB, MUTPB = 0.5, 0.2  # Crossover and mutation probabilities

# Register genetic operators with custom parameters
toolbox.register("mate", tools.cxUniform, indpb=0.5)  # Uniform crossover
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.05)  # Gaussian mutation for real values
toolbox.register("select", tools.selTournament, tournsize=3)  # Tournament selection

# Hall of Fame to keep track of the best individuals
hof = tools.HallOfFame(1)  # Save the best individual

# Run Genetic Algorithm with elite preservation
for gen in range(NGEN):
    offspring = algorithms.varAnd(population, toolbox, cxpb=CXPB, mutpb=MUTPB)
    fits = map(toolbox.evaluate, offspring)

    # Evaluate all offspring and assign fitness
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit

    # Elitism: Add the best individuals back into the population
    elite = tools.selBest(population, k=1)
    population = toolbox.select(offspring, k=len(population) - 1) + elite

    # Log best fitness in generation
    print(f"Generation {gen} - Best Accuracy: {max(ind.fitness.values for ind in population)}")

    # Update the hall of fame with the best individuals
    hof.update(population)

print("Best Individual: ", hof[0])


## Identify the best individual from the final population.


best_individual: Selects the best individual.


best_trees: Gets the trees corresponding to the best individual.

In [ ]:

best_individual = tools.selBest(population, k=1)[0]
best_trees = [tree for i, tree in enumerate(initial_rf.estimators_) if best_individual[i] == 1]

# Create the pruned Random Forest with the best individual
pruned_rf = RandomForestClassifier(n_estimators=len(best_trees), random_state=42)
pruned_rf.estimators_ = best_trees
pruned_rf.classes_ = initial_rf.classes_
pruned_rf.n_classes_ = initial_rf.n_classes_
pruned_rf.n_outputs_ = initial_rf.n_outputs_

# Final evaluation
y_pred = pruned_rf.predict(X_test_top10)  # Use the test data with top 10 features
print("Accuracy of pruned Random Forest:", accuracy_score(y_test, y_pred))


## Visualize feature importance.


importances: Gets feature importances from the pruned Random Forest.


indices: Sorts feature indices by importance.


plt.figure: Creates a new figure.


sns.barplot: Plots a bar plot of feature importances.


plt.show(): Displays the plot.

In [ ]:
# # Feature Importance Plot
# importances = pruned_rf.feature_importances_
# indices = np.argsort(importances)[::-1]

# # plt.figure(figsize=(10, 30))
# plt.figure(figsize=(10, 26))
# sns.barplot(x=importances[indices], y=features.columns[indices])
# plt.title('Feature Importances')
# plt.xlabel('Importance')
# plt.ylabel('Features')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Feature Importance Plot
importances = pruned_rf.feature_importances_
indices = np.argsort(importances)[::-1]

# Select the top 20 features

top_indices = indices[:top_n]
top_features = features.columns[top_indices]

# Print out the names of the top 20 features
print(f"Top {top_n} Features:")
for feature in top_features:
    print(feature)

# Plot
plt.figure(figsize=(10, 10))  # Adjusted figure size for better readability
sns.barplot(x=importances[top_indices], y=top_features)
plt.title(f'Top {top_n} Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Features')
plt.show()


## Save visualizations of each tree in the pruned Random Forest.


os.makedirs: Creates the directory "Tree Branches" if it doesn't exist.


The for loop iterates over each tree, exports it to DOT format, converts it to a graph, and saves it as a PNG file in the "Tree Branches" directory.


print: Confirms that the tree branches have been saved.

## TURN ME BACK ON

In [ ]:
import os
import pydotplus
from sklearn.tree import export_graphviz

# Ensure the directory exists for saving the tree branches
os.makedirs("Tree Branches", exist_ok=True)

# Assuming 'X_train_top10' is the DataFrame with the top 10 features
# Extract the correct feature names for the top 10 features
top_10_feature_names = X_train.columns[top_10_indices]

for i, tree in enumerate(pruned_rf.estimators_):
    dot_data = export_graphviz(tree, 
                               out_file=None, 
                               feature_names=top_10_feature_names,  # Use the top 10 feature names
                               class_names=[str(cls) for cls in pruned_rf.classes_], 
                               filled=True, 
                               rounded=True, 
                               special_characters=True)
    
    # Convert to graph and save as PNG
    graph = pydotplus.graph_from_dot_data(dot_data)
    graph.write_png(f"Tree Branches/tree_{i}.png")

print("Tree branches saved in the 'Tree Branches' folder.")


In [ ]:
import os
from sklearn.tree import export_text

# Create output directory
os.makedirs("RF_Tree_Text", exist_ok=True)

for i, tree in enumerate(pruned_rf.estimators_):
    
    tree_text = export_text(
        tree,
        feature_names=list(top_10_feature_names),
        show_weights=True
    )
    
    file_path = f"RF_Tree_Text/tree_{i}.txt"
    
    with open(file_path, "w") as f:
        f.write(tree_text)

print("All trees exported as readable text.")

## Guide to tune the Random Forest algorithm effectively:

1. **Understanding the Parameters**:  Key parameters include 
- the number of trees (`n_estimators`)
- maximum depth of trees (`max_depth`)
- minimum number of samples required to split an internal node (`min_samples_split`)
- minimum number of samples required to be a leaf node (`min_samples_leaf`). 

2. **Define a Parameter Grid**: Try from a variety of parameters for tuning, like the below:
    ```python
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    ```

3. **Perform Cross-Validation**: Use techniques like k-fold cross-validation to evaluate the performance of the model for each combination of hyperparameters which helps avoid overfitting and provides a more reliable estimate of model performance.

4. **Hyperparameter Tuning**: Use techniques like grid search or random search to find the optimal combination of hyperparameters. Grid search exhaustively searches through all possible combinations of hyperparameters, while random search randomly selects combinations to evaluate. Grid search is more exhaustive but can be computationally expensive, especially for a large parameter grid.

5. **Evaluate Performance Metrics**: Choose appropriate performance metrics to evaluate the model's performance. Common metrics for classification tasks include accuracy, precision, recall, F1-score, and area under the ROC curve (AUC-ROC). Select the metric(s) that are most relevant to your problem and use them to evaluate the performance of different parameter combinations.

6. **Visualize Results**: Visualize the results of hyperparameter tuning to understand the relationship between hyperparameters and model performance. You can use tools like heatmaps, line plots, or bar plots to visualize how different hyperparameters affect the model's performance.

7. **Select the Best Model**: Once you have evaluated the performance of different parameter combinations, select the model with the best performance based on your chosen evaluation metric(s).

8. **Validate the Final Model**: Finally, validate the performance of the selected model on a separate test set to ensure that it generalizes well to unseen data. Evaluate the model using the chosen performance metrics and compare the results to those obtained during hyperparameter tuning.

By following these steps, you can effectively tune the Random Forest algorithm and improve its performance for your specific problem.

# Interpreting Tree Branches

Interpreting decision tree branches can provide insights into how the model makes decisions based on the features. Here's a detailed guide on how to interpret the tree branches, along with code to save and visualize the branches:

Step-by-Step Guide to Interpret Tree Branches
Understand the Basics of Decision Trees:

Nodes: Each internal node represents a feature (or attribute), and a decision is made based on its value.
Edges: The branches or edges connect nodes, representing the outcome of the decision at each node.
Leaves: The leaf nodes represent the final decision or classification.
Tree Visualization:

Graph Representation: Visualize the decision tree using graph representations like DOT format, which can be rendered into images.
Feature Names: Each node shows the feature used for the split.
Thresholds: The threshold value of the feature used for the split.
Samples: The number of training samples at that node.
Classes: The distribution of classes at the node.
Gini Impurity: A measure of impurity or homogeneity at that node.
Code to Save and Visualize Tree Branches:

Save each tree branch to an image file using export_graphviz and pydotplus.
Use the saved images to analyze and interpret the decision paths.


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score
import matplotlib.pyplot as plt

# Predict the labels for the test set using only the top 10 important features
y_pred = pruned_rf.predict(X_test_top10)

# Create a confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pruned_rf.classes_)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

# Final evaluation - Accuracy score
print("Accuracy of pruned Random Forest:", accuracy_score(y_test, y_pred))
print(file_name)


# ROC Curve

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

# Predict probabilities for each class using the pruned random forest with top 10 features
y_probs = pruned_rf.predict_proba(X_test_top10)

# Compute ROC curve and ROC area for each class
fpr = {}
tpr = {}
roc_auc = {}

n_classes = len(pruned_rf.classes_)

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test == pruned_rf.classes_[i], y_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot ROC curve for each class
plt.figure(figsize=(10, 8))

colors = ['blue', 'red', 'green', 'orange']  # Define colors as needed

for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label='ROC curve of class {0} (area = {1:0.2f})'
                   ''.format(pruned_rf.classes_[i], roc_auc[i]))

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()


In [ ]:
from sklearn.metrics import f1_score

# Compute F1 score for each class
f1 = f1_score(y_test, y_pred, average=None, labels=pruned_rf.classes_)

# Print the F1 scores for each class
for i, class_name in enumerate(pruned_rf.classes_):
    print(f'F1 score for class {class_name}: {f1[i]:.2f}')


In [ ]:
print('Done')